# Notebook 08 — Multi-Source Retrieval Fusion (MVP)

**Run after Notebooks 02–07.** Combine 30-day time-decayed popularity (Notebook 04), co-occurrence Item-CF (05), FAISS IVF `nprobe=16` (07), and **point-in-time category-aware retrieval** (02–03). Use fixed, untuned Reciprocal Rank Fusion (RRF), then append unseen popularity candidates only when needed to reach Top-500. Evaluate **RRF Top-100/300/500**, raw-source candidate-union oracle coverage, source overlaps and unique positive hits, user/catalog coverage, and batch timings.

**Important:** Source Top-100 files do not become Top-500 by themselves. Category contributes up to 200, and the extra popularity pool provides up to 500 fallback candidates. Report the *observed* candidate depth; a fallback candidate is not evidence of personalization. No future event or property data may enter retrieval. Test is for confirmation, not tuning.


## 1. Setup

In the active Windows Conda/Jupyter kernel: `%pip install duckdb pandas numpy pyarrow matplotlib`. Close any other Notebook holding a write connection to `notebook02.duckdb`.

In [2]:
from pathlib import Path
import json, time, math
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display

cwd=Path.cwd().resolve()
ROOT=next((p for p in (cwd,*cwd.parents) if (p/'data'/'processed'/'notebook02.duckdb').exists()),None)
assert ROOT is not None, 'Cannot find data/processed/notebook02.duckdb. Open notebook from project folder.'
OUT=ROOT/'data'/'processed'; REPORT=ROOT/'data'/'reports'; REPORT.mkdir(parents=True,exist_ok=True)
DB=OUT/'notebook02.duckdb'
con=duckdb.connect(str(DB))
con.execute("SET memory_limit='4GB'")
con.execute('SET threads=4')
splits=('train','validation','test')
def q(path):return "'"+str(path).replace("'","''")+"'"
def table_exists(name):
    return bool(con.execute('SELECT COUNT(*) FROM information_schema.tables WHERE table_name=?',[name]).fetchone()[0])
print('Project:',ROOT,'DuckDB:',duckdb.__version__)


Project: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks DuckDB: 1.5.5


## 2. Input contract and fixed configuration

`{split}_item_features` and `{split}_user_category` were computed from historical data and item-property snapshots at or before the split cutoff. Do **not** recompute categories from raw property files without an `as-of` filter. Category ancestry comes from the static category tree; the category assignment is time-varying.

For reproducibility the RRF weights are predeclared: CF 1.0, FAISS 0.8, Category 0.8, Popularity 0.5. These are **heuristic**, not learned or optimized on test. `RRF_K=60`. The fallback pool is not included as an RRF vote; it is appended only after all fused primary candidates, ordered by historical time-decayed popularity.

In [3]:
TOP_K=500
KS=(20,50,100,300,500)
RRF_K=60
WEIGHTS={'cf':1.0,'faiss':0.8,'category':0.8,'popularity':0.5}
CATEGORY_PER_USER=200
MAX_CATEGORIES_PER_USER=5
ITEMS_PER_CATEGORY=120
FALLBACK_K=500
DAY=86_400_000
RUNS=[]; METRICS=[]; SOURCE_METRICS=[]; OVERLAPS=[]; UNIQUE_HITS=[]; SANITY=[]; UNION_METRICS=[]; PIT_CHECKS=[]

required=['category_tree','category_ancestry']
for s in splits:
    required += [f'{s}_positives',f'{s}_history',f'{s}_enriched_examples',
                 f'{s}_item_features',f'{s}_user_category',f'{s}_pop_scores']
missing=[x for x in required if not table_exists(x)]
assert not missing, f'Missing Notebook 02–04 tables: {missing}'
for s in splits:
    for label,p in {'popularity':OUT/f'{s}_pop_decayed_top100.parquet',
                    'cf':OUT/f'{s}_cf_cooccurrence_top100.parquet',
                    'faiss':OUT/f'{s}_faiss_ivf_np16_top100.parquet'}.items():
        assert p.is_file(),f'Missing {label}: {p}; run Notebook 04/05/07 first.'
    cutoff=con.execute(f'SELECT MIN(cutoff_ts),MAX(cutoff_ts) FROM {s}_enriched_examples').fetchone()
    assert cutoff[0]==cutoff[1] and cutoff[0] is not None
    assert con.execute(f'SELECT COUNT(*) FROM events_raw WHERE ts >= ? AND ts < ?',[cutoff[0]-30*DAY,cutoff[0]]).fetchone()[0]>0
    # Notebook 02 category mapping is derived solely from props_all rows with ts<=cutoff.
    assert table_exists(f'{s}_item_props')
    invalid=con.execute(f'''SELECT COUNT(*) FROM {s}_item_features f
      LEFT JOIN category_tree t ON f.category_id=t.category_id
      WHERE f.category_id IS NOT NULL AND t.category_id IS NULL''').fetchone()[0]
    assert invalid==0
    PIT_CHECKS.append(dict(split=s,cutoff_ts=int(cutoff[0]),category_assignment_source=f'{s}_item_features',
                           known_category_ids_valid=True,source_created_in_notebook02_with_ts_le_cutoff=True))
print('All required tables and Parquet inputs found.')


All required tables and Parquet inputs found.


## 3. Register existing three Top-100 candidate sources

Input Parquet columns are `user_id`, `item_id`, `retrieval_score`, `retrieval_rank`, `retrieval_source`. Preserve the original ranks; source scores have incompatible scales and **must not** be added directly.

In [4]:
for s in splits:
    for label,filename in [('popularity',f'{s}_pop_decayed_top100.parquet'),
                           ('cf',f'{s}_cf_cooccurrence_top100.parquet'),
                           ('faiss',f'{s}_faiss_ivf_np16_top100.parquet')]:
        table=f'{s}_fusion_source_{label}'
        con.execute(f'''CREATE OR REPLACE TABLE {table} AS
           SELECT user_id::BIGINT AS user_id,item_id::BIGINT AS item_id,
                  retrieval_rank::INTEGER AS retrieval_rank,
                  retrieval_score::DOUBLE AS retrieval_score,
                  '{label}'::VARCHAR AS retrieval_source
           FROM read_parquet({q(OUT/filename)})''')
        assert con.execute(f'''SELECT COUNT(*) FROM {table} c
          JOIN {s}_history h ON c.user_id=h.user_id AND c.item_id=h.item_id''').fetchone()[0]==0
        assert con.execute(f'''SELECT COUNT(*) FROM (
          SELECT user_id,item_id,COUNT(*) n FROM {table} GROUP BY 1,2 HAVING COUNT(*)>1)''').fetchone()[0]==0
        print(s,label,con.execute(f'SELECT COUNT(*),COUNT(DISTINCT user_id) FROM {table}').fetchone())


train popularity (411000, 4110)
train cf (150451, 3417)
train faiss (328500, 3285)
validation popularity (365700, 3657)
validation cf (123940, 2921)
validation faiss (289000, 2890)
test popularity (357200, 3572)
test cf (114824, 2830)
test faiss (285800, 2858)


## 4. Category-Aware Retrieval — genuine use of both property parts and category tree

Item category comes from Notebook 02's latest known `categoryid` across both `item_properties_part1.csv` and `item_properties_part2.csv`, restricted to property timestamps `<= cutoff`. Notebook 03's `category_ancestry` supplies `root_category_id`. User affinity is computed from 30-day pre-cutoff interactions, joined to that same cutoff's item category. Rank each user's five strongest categories and recommend up to 200 popular *unseen* items in those categories. Score combines historical category affinity and item interaction count; no future labels.

We also permit **root-category matching** at lower weight to exploit the tree hierarchy. The root is derived from the item's known category, not guessed from unknown properties.

In [5]:
def build_category(split):
    s=split;t0=time.perf_counter()
    # Historical item popularity + point-in-time category. Only observed pre-cutoff items are eligible.
    con.execute(f'''CREATE OR REPLACE TEMP TABLE tmp_{s}_cat_items AS
      SELECT p.item_id, f.category_id,a.root_category_id,
             p.item_30d_events::DOUBLE AS popularity
      FROM {s}_item_popularity p
      JOIN {s}_item_features f USING(item_id)
      JOIN category_ancestry a ON f.category_id=a.category_id
      WHERE f.category_id IS NOT NULL AND p.item_30d_events>0''')
    con.execute(f'''CREATE OR REPLACE TEMP TABLE tmp_{s}_cat_users AS
      WITH prefs AS (
        SELECT uc.user_id,uc.category_id,a.root_category_id,
               uc.user_category_30d_events::DOUBLE AS affinity,
               ROW_NUMBER() OVER(PARTITION BY uc.user_id ORDER BY uc.user_category_30d_events DESC,uc.category_id) AS preference_rank
        FROM {s}_user_category uc
        JOIN category_ancestry a USING(category_id)
        JOIN (SELECT DISTINCT user_id FROM {s}_positives) u USING(user_id)
        WHERE uc.user_category_30d_events>0
      ) SELECT * FROM prefs WHERE preference_rank<={MAX_CATEGORIES_PER_USER}''')
    # Bound the per-category item pool. Distinct categories may share a root.
    con.execute(f'''CREATE OR REPLACE TEMP TABLE tmp_{s}_cat_pool AS
      WITH ranked AS (
        SELECT *,ROW_NUMBER() OVER(PARTITION BY category_id ORDER BY popularity DESC,item_id) AS item_rank
        FROM tmp_{s}_cat_items
      ) SELECT * FROM ranked WHERE item_rank<={ITEMS_PER_CATEGORY}''')
    # Bound broad root matches separately; otherwise a root with many categories
    # can create a very large user-item cross product on a Windows laptop.
    con.execute(f'''CREATE OR REPLACE TEMP TABLE tmp_{s}_root_pool AS
      WITH ranked AS (
        SELECT *,ROW_NUMBER() OVER(PARTITION BY root_category_id ORDER BY popularity DESC,item_id) AS root_rank
        FROM tmp_{s}_cat_items
      ) SELECT * FROM ranked WHERE root_rank<=120''')
    # Exact category and broader root-category matches are two candidate paths.
    con.execute(f'''CREATE OR REPLACE TABLE {s}_fusion_source_category AS
      WITH exact_matches AS (
        SELECT u.user_id,i.item_id, u.affinity * (1.0+LN(1.0+i.popularity)) AS score
        FROM tmp_{s}_cat_users u JOIN tmp_{s}_cat_pool i ON u.category_id=i.category_id
      ), root_matches AS (
        SELECT u.user_id,i.item_id,0.20*u.affinity * (1.0+LN(1.0+i.popularity)) AS score
        FROM tmp_{s}_cat_users u JOIN tmp_{s}_root_pool i ON u.root_category_id=i.root_category_id
        WHERE u.category_id<>i.category_id
      ), grouped AS (
        SELECT user_id,item_id,SUM(score) AS retrieval_score
        FROM (SELECT * FROM exact_matches UNION ALL SELECT * FROM root_matches)
        GROUP BY 1,2
      ), unseen AS (
        SELECT g.* FROM grouped g
        WHERE NOT EXISTS (SELECT 1 FROM {s}_history h
          WHERE h.user_id=g.user_id AND h.item_id=g.item_id)
      ), ranked AS (
        SELECT *,ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY retrieval_score DESC,item_id) AS retrieval_rank
        FROM unseen
      ) SELECT user_id,item_id,retrieval_rank::INTEGER AS retrieval_rank,
               retrieval_score::DOUBLE AS retrieval_score,'category'::VARCHAR AS retrieval_source
        FROM ranked WHERE retrieval_rank<={CATEGORY_PER_USER}''')
    elapsed=time.perf_counter()-t0
    n,users=con.execute(f'SELECT COUNT(*),COUNT(DISTINCT user_id) FROM {s}_fusion_source_category').fetchone()
    print(s,'category',n,'rows',users,'users',round(elapsed,3),'sec')
    return dict(split=s,stage='category_build',seconds=elapsed,rows=int(n),covered_users=int(users))

for s in splits:RUNS.append(build_category(s))


train category 747490 rows 3822 users 0.417 sec
validation category 657694 rows 3386 users 0.355 sec
test category 640288 rows 3298 users 0.349 sec


## 5. Extra time-decayed popularity pool for fallback

Notebook 04 exports only Top-100. Generate a **new Top-500 per evaluation user**, using its existing 7-day half-life pre-cutoff `decayed_score` and excluding the full 30-day seen history. The pool size is `500 + max_seen`, so filtering cannot accidentally reduce Top-500 when enough eligible catalog items exist. Fallback items are *not* extra RRF votes.

In [6]:
def build_fallback(s):
    t0=time.perf_counter()
    max_seen=int(con.execute(f'''SELECT COALESCE(MAX(n),0) FROM (
      SELECT COUNT(*) n FROM {s}_history h
      JOIN (SELECT DISTINCT user_id FROM {s}_positives) u USING(user_id)
      GROUP BY h.user_id)''').fetchone()[0])
    catalog=int(con.execute(f'SELECT COUNT(*) FROM {s}_pop_scores').fetchone()[0])
    pool=min(catalog,FALLBACK_K+max_seen)
    con.execute(f'''CREATE OR REPLACE TABLE {s}_fusion_fallback AS
      WITH top_items AS (
        SELECT item_id,decayed_score::DOUBLE AS retrieval_score
        FROM {s}_pop_scores ORDER BY decayed_score DESC,item_id LIMIT {pool}
      ), eligible AS (
        SELECT u.user_id,p.item_id,p.retrieval_score
        FROM (SELECT DISTINCT user_id FROM {s}_positives) u CROSS JOIN top_items p
        WHERE NOT EXISTS(SELECT 1 FROM {s}_history h
          WHERE h.user_id=u.user_id AND h.item_id=p.item_id)
      ), ranked AS (
        SELECT *,ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY retrieval_score DESC,item_id) AS retrieval_rank
        FROM eligible
      ) SELECT user_id,item_id,retrieval_rank::INTEGER AS retrieval_rank,
               retrieval_score,'fallback'::VARCHAR AS retrieval_source
        FROM ranked WHERE retrieval_rank<={FALLBACK_K}''')
    n,users=con.execute(f'SELECT COUNT(*),COUNT(DISTINCT user_id) FROM {s}_fusion_fallback').fetchone()
    seconds=time.perf_counter()-t0
    print(s,'fallback',n,'rows',users,'users',round(seconds,3),'sec')
    return dict(split=s,stage='fallback_build',seconds=seconds,rows=int(n),covered_users=int(users),pool_size=pool)

for s in splits:RUNS.append(build_fallback(s))


train fallback 2055000 rows 4110 users 1.251 sec
validation fallback 1828500 rows 3657 users 0.957 sec
test fallback 1786000 rows 3572 users 0.811 sec


## 6. Source unions and fixed-weight RRF

For `(user,item)` appearing in multiple sources, sum `weight/(60+source_rank)`. Preserve `source_count` and individual source flags for future ranking features. **RRF Top-500** first orders the distinct candidates from the four primary sources, then fills remaining slots from the fallback pool. The fallback score is deliberately separate and does not compete with RRF scores. We also export the raw four-source candidate union, capped only when measuring union@K (an oracle pool diagnostic, **not** a deployable ranked recommendation).

In [7]:
def fuse(s):
    t0=time.perf_counter()
    branches=[]
    for label in WEIGHTS:
        branches.append(f"SELECT user_id,item_id,retrieval_rank,'{label}'::VARCHAR AS source FROM {s}_fusion_source_{label}")
    con.execute(f'''CREATE OR REPLACE TABLE {s}_fusion_all_sources AS {' UNION ALL '.join(branches)}''')
    case=' '.join(f"WHEN source='{name}' THEN {weight}" for name,weight in WEIGHTS.items())
    con.execute(f'''CREATE OR REPLACE TABLE {s}_fusion_rrf_scored AS
      SELECT user_id,item_id,
        SUM((CASE {case} ELSE 0 END) / ({RRF_K}+retrieval_rank))::DOUBLE AS rrf_score,
        COUNT(DISTINCT source)::INTEGER AS source_count,
        MAX((source='popularity')::INTEGER)::INTEGER AS from_popularity,
        MAX((source='cf')::INTEGER)::INTEGER AS from_cf,
        MAX((source='faiss')::INTEGER)::INTEGER AS from_faiss,
        MAX((source='category')::INTEGER)::INTEGER AS from_category
      FROM {s}_fusion_all_sources GROUP BY 1,2''')
    con.execute(f'''CREATE OR REPLACE TABLE {s}_fusion_rrf_ranked AS
      SELECT *,ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY rrf_score DESC,source_count DESC,item_id) AS primary_rank
      FROM {s}_fusion_rrf_scored''')
    # Reserve all primary candidates (up to 500) ahead of fallback. Never mix score scales.
    con.execute(f'''CREATE OR REPLACE TABLE {s}_fusion_candidates AS
      WITH primary_candidates AS (
        SELECT user_id,item_id,rrf_score AS retrieval_score,source_count,
               from_popularity,from_cf,from_faiss,from_category,
               0::INTEGER AS is_fallback,primary_rank AS order_rank,0::INTEGER AS tier
        FROM {s}_fusion_rrf_ranked WHERE primary_rank<={TOP_K}
      ), fallback_candidates AS (
        SELECT f.user_id,f.item_id,NULL::DOUBLE AS retrieval_score,
               0::INTEGER AS source_count,0::INTEGER AS from_popularity,
               0::INTEGER AS from_cf,0::INTEGER AS from_faiss,0::INTEGER AS from_category,
               1::INTEGER AS is_fallback,f.retrieval_rank AS order_rank,1::INTEGER AS tier
        FROM {s}_fusion_fallback f
        WHERE NOT EXISTS (SELECT 1 FROM primary_candidates p
          WHERE p.user_id=f.user_id AND p.item_id=f.item_id)
      ), ranked AS (
        SELECT *,ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY tier,order_rank,item_id) AS retrieval_rank
        FROM (SELECT * FROM primary_candidates UNION ALL SELECT * FROM fallback_candidates)
      ) SELECT user_id,item_id,retrieval_rank::INTEGER AS retrieval_rank,
               retrieval_score,source_count,from_popularity,from_cf,from_faiss,from_category,
               is_fallback,'fusion_rrf'::VARCHAR AS retrieval_source
        FROM ranked WHERE retrieval_rank<={TOP_K}''')
    elapsed=time.perf_counter()-t0
    n,users,fb=con.execute(f'''SELECT COUNT(*),COUNT(DISTINCT user_id),SUM(is_fallback)
       FROM {s}_fusion_candidates''').fetchone()
    path=OUT/f'{s}_fusion_rrf_top500.parquet'
    con.execute(f'COPY {s}_fusion_candidates TO {q(path)} (FORMAT PARQUET)')
    unionpath=OUT/f'{s}_fusion_raw_source_union.parquet'
    con.execute(f'COPY {s}_fusion_rrf_scored TO {q(unionpath)} (FORMAT PARQUET)')
    print(s,'fusion rows',n,'covered',users,'fallback rows',fb,'seconds',round(elapsed,3))
    return dict(split=s,stage='rrf_fusion',seconds=elapsed,candidate_rows=int(n),covered_users=int(users),
                fallback_rows=int(fb),candidates_path=path.name,raw_union_path=unionpath.name)

for s in splits:RUNS.append(fuse(s))


train fusion rows 2055000 covered 4110 fallback rows 573624 seconds 1.845
validation fusion rows 1828500 covered 3657 fallback rows 527209 seconds 1.708
test fusion rows 1786000 covered 3572 fallback rows 524774 seconds 1.734


## 7. Sanity checks: duplicates, seen history, rank integrity, coverage and temporal invariants

In [8]:
for s in splits:
    t=f'{s}_fusion_candidates'
    n,covered,dups=con.execute(f'''SELECT COUNT(*),COUNT(DISTINCT user_id),
       COUNT(*)-COUNT(DISTINCT (user_id,item_id)) FROM {t}''').fetchone()
    seen=con.execute(f'''SELECT COUNT(*) FROM {t} c JOIN {s}_history h
       ON c.user_id=h.user_id AND c.item_id=h.item_id''').fetchone()[0]
    invalid=con.execute(f'''SELECT COUNT(*) FROM (
       SELECT user_id,COUNT(*) n,MIN(retrieval_rank) lo,MAX(retrieval_rank) hi
       FROM {t} GROUP BY user_id) WHERE lo<>1 OR hi<>n OR n>{TOP_K}''').fetchone()[0]
    non_eval=con.execute(f'''SELECT COUNT(*) FROM {t} c WHERE NOT EXISTS(
       SELECT 1 FROM {s}_positives p WHERE p.user_id=c.user_id)''').fetchone()[0]
    eval_users=con.execute(f'SELECT COUNT(DISTINCT user_id) FROM {s}_positives').fetchone()[0]
    short=con.execute(f'''SELECT COUNT(*) FROM (SELECT user_id,COUNT(*) n
      FROM {t} GROUP BY user_id) WHERE n<{TOP_K}''').fetchone()[0]
    assert dups==seen==invalid==non_eval==0,(s,dups,seen,invalid,non_eval)
    assert covered==eval_users,(s,covered,eval_users)
    SANITY.append(dict(split=s,rows=int(n),evaluation_users=int(eval_users),covered_users=int(covered),
                       duplicate_pairs=int(dups),seen_item_leakage=int(seen),bad_ranks=int(invalid),
                       non_evaluation_users=int(non_eval),users_with_fewer_than_500=int(short)))
print(pd.DataFrame(SANITY).to_string(index=False))


     split    rows  evaluation_users  covered_users  duplicate_pairs  seen_item_leakage  bad_ranks  non_evaluation_users  users_with_fewer_than_500
     train 2055000              4110           4110                0                  0          0                     0                          0
validation 1828500              3657           3657                0                  0          0                     0                          0
      test 1786000              3572           3572                0                  0          0                     0                          0


## 8. Metrics: RRF ranking, individual sources, and raw union diagnostics

Micro Recall@K = eligible future positive pairs retrieved in Top-K / all eligible future positive pairs. HitRate@K = fraction of evaluation users with at least one hit. **Union coverage is an oracle candidate-pool diagnostic:** it checks whether a future positive appears *anywhere* in the four-source union (and optionally fallback), without assuming the union is ranked. The raw union may have more than 500 items; therefore report its actual size and label union recall as *unranked* rather than presenting it as Recall@500.

In [9]:
def eval_ranked(s,table,method,ks=KS):
    total,users=map(int,con.execute(f'SELECT COUNT(*),COUNT(DISTINCT user_id) FROM {s}_positives').fetchone())
    catalog=int(con.execute(f'SELECT COUNT(*) FROM {s}_pop_scores').fetchone()[0])
    result=[]
    for k in ks:
        hits,hit_users,items,covered=con.execute(f'''WITH top AS (
          SELECT user_id,item_id FROM {table} WHERE retrieval_rank<={int(k)}
        ), matched AS (
          SELECT p.user_id,p.item_id FROM {s}_positives p JOIN top t
          ON p.user_id=t.user_id AND p.item_id=t.item_id
        ) SELECT (SELECT COUNT(*) FROM matched),
          (SELECT COUNT(DISTINCT user_id) FROM matched),
          (SELECT COUNT(DISTINCT item_id) FROM top),
          (SELECT COUNT(DISTINCT user_id) FROM top)''').fetchone()
        result.append(dict(split=s,method=method,k=int(k),eligible_positive_pairs=total,
                           evaluation_users=users,hits=int(hits),recall_micro=hits/total if total else None,
                           hit_rate=hit_users/users if users else None,
                           covered_users=int(covered),user_coverage=covered/users if users else None,
                           unique_recommended_items=int(items),catalog_coverage=items/catalog if catalog else None))
    return result

for s in splits:
    METRICS.extend(eval_ranked(s,f'{s}_fusion_candidates','fusion_rrf'))
    for source in WEIGHTS:
        SOURCE_METRICS.extend(eval_ranked(s,f'{s}_fusion_source_{source}',source,ks=(20,50,100)))
    # Actual raw source union (unranked), with no artificial K label.
    total=int(con.execute(f'SELECT COUNT(*) FROM {s}_positives').fetchone()[0])
    hit,union_pairs,unique_items=con.execute(f'''SELECT
      (SELECT COUNT(*) FROM {s}_positives p JOIN {s}_fusion_rrf_scored c
       ON p.user_id=c.user_id AND p.item_id=c.item_id),
      (SELECT COUNT(*) FROM {s}_fusion_rrf_scored),
      (SELECT COUNT(DISTINCT item_id) FROM {s}_fusion_rrf_scored)''').fetchone()
    UNION_METRICS.append(dict(split=s,pool='four_primary_sources_unranked',
       candidate_pairs=int(union_pairs),unique_items=int(unique_items),hits=int(hit),
       eligible_positive_pairs=total,oracle_union_recall=hit/total if total else None))

metrics_df=pd.DataFrame(METRICS);source_df=pd.DataFrame(SOURCE_METRICS);union_df=pd.DataFrame(UNION_METRICS)
display(metrics_df[['split','k','recall_micro','hit_rate','user_coverage','catalog_coverage']])
display(union_df)


,split,k,recall_micro,hit_rate,user_coverage,catalog_coverage
0,train,20,0.076052,0.136010,1.0,0.184975
1,train,50,0.114399,0.190268,1.0,0.291063
2,train,100,0.142918,0.227737,1.0,0.371714
3,train,300,0.197607,0.287591,1.0,0.478949
4,train,500,0.227622,0.324574,1.0,0.484803
5,validation,20,0.068182,0.131255,1.0,0.183759
6,validation,50,0.102772,0.178015,1.0,0.292598
7,validation,100,0.131375,0.218485,1.0,0.374782
8,validation,300,0.191353,0.296965,1.0,0.483562
9,validation,500,0.221286,0.327591,1.0,0.490012


,split,pool,candidate_pairs,unique_items,hits,eligible_positive_pairs,oracle_union_recall
0,train,four_primary_sources_unranked,1481376,60454,2049,9362,0.218863
1,validation,four_primary_sources_unranked,1301291,57668,1888,9020,0.209313
2,test,four_primary_sources_unranked,1261226,54976,1679,8214,0.204407


## 9. Source overlap and exclusive positive hits

Pairwise overlap uses distinct `(user_id,item_id)` across sources (the category source can supply 200, others 100). Exclusive hits count **future-positive pairs retrieved by only that one primary source**. This is a diagnostic, not causal attribution.

In [10]:
for s in splits:
    for i,a in enumerate(WEIGHTS):
        for b in list(WEIGHTS)[i+1:]:
            na,nb,intersection=con.execute(f'''SELECT
              (SELECT COUNT(*) FROM {s}_fusion_source_{a}),
              (SELECT COUNT(*) FROM {s}_fusion_source_{b}),
              (SELECT COUNT(*) FROM {s}_fusion_source_{a} x JOIN {s}_fusion_source_{b} y
               ON x.user_id=y.user_id AND x.item_id=y.item_id)''').fetchone()
            OVERLAPS.append(dict(split=s,source_a=a,source_b=b,pairs_a=int(na),pairs_b=int(nb),
                intersect_pairs=int(intersection),jaccard_pairs=intersection/(na+nb-intersection) if na+nb-intersection else None))
    # One row per positive pair in primary union; source_count=1 means unique to one source.
    for source in WEIGHTS:
        hits,exclusive=con.execute(f'''SELECT COUNT(*),
           COUNT(*) FILTER(WHERE c.source_count=1)
           FROM {s}_positives p JOIN {s}_fusion_rrf_scored c
           ON p.user_id=c.user_id AND p.item_id=c.item_id
           WHERE c.from_{source}=1''').fetchone()
        UNIQUE_HITS.append(dict(split=s,source=source,positive_hits=int(hits),exclusive_positive_hits=int(exclusive)))

ov_df=pd.DataFrame(OVERLAPS);unique_df=pd.DataFrame(UNIQUE_HITS)
display(ov_df[ov_df.split=='validation'])
display(unique_df[unique_df.split=='validation'])


,split,source_a,source_b,pairs_a,pairs_b,intersect_pairs,jaccard_pairs
6,validation,cf,faiss,123940,289000,21683,0.055419
7,validation,cf,category,123940,657694,38215,0.051404
8,validation,cf,popularity,123940,365700,4899,0.010106
9,validation,faiss,category,289000,657694,27959,0.030432
10,validation,faiss,popularity,289000,365700,1352,0.002069
11,validation,category,popularity,657694,365700,55418,0.057251


,split,source,positive_hits,exclusive_positive_hits
4,validation,cf,788,194
5,validation,faiss,351,73
6,validation,category,1408,725
7,validation,popularity,331,152


## 10. Ablation: compare fixed RRF with/without category source

To avoid falsely claiming that category improves fusion, perform a lightweight validation ablation. This is a *fixed heuristic comparison*; it does not tune weights. When category is excluded, append the same popularity fallback. The category contribution to *candidate union* and to Top-K ranking can differ.

In [11]:
def build_no_category(s):
    con.execute(f'''CREATE OR REPLACE TABLE {s}_fusion_no_category AS
      WITH scores AS (
        SELECT user_id,item_id,
          SUM((CASE WHEN source='cf' THEN {WEIGHTS['cf']}
                    WHEN source='faiss' THEN {WEIGHTS['faiss']}
                    WHEN source='popularity' THEN {WEIGHTS['popularity']}
                    ELSE 0 END)/({RRF_K}+retrieval_rank)) AS rrf_score
        FROM {s}_fusion_all_sources WHERE source<>'category' GROUP BY 1,2
      ), primary_ranked AS (
        SELECT user_id,item_id,rrf_score,
          ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY rrf_score DESC,item_id) AS r
        FROM scores
      ), joined AS (
        SELECT user_id,item_id,0::INTEGER AS tier,r AS ordering FROM primary_ranked WHERE r<={TOP_K}
        UNION ALL
        SELECT f.user_id,f.item_id,1::INTEGER AS tier,f.retrieval_rank AS ordering
        FROM {s}_fusion_fallback f
        WHERE NOT EXISTS (SELECT 1 FROM primary_ranked p WHERE p.user_id=f.user_id AND p.item_id=f.item_id)
      ), ranked AS (
        SELECT user_id,item_id,ROW_NUMBER() OVER(PARTITION BY user_id ORDER BY tier,ordering,item_id) AS retrieval_rank
        FROM joined
      ) SELECT * FROM ranked WHERE retrieval_rank<={TOP_K}''')
    return eval_ranked(s,f'{s}_fusion_no_category','fusion_no_category',ks=(100,300,500))

ABLATION=[]
for s in splits:ABLATION.extend(build_no_category(s))
abl_df=pd.DataFrame(ABLATION)
display(pd.concat([metrics_df[metrics_df.k.isin([100,300,500])],abl_df],ignore_index=True)
        .query("split=='validation'")[['method','k','recall_micro','hit_rate','catalog_coverage']])


,method,k,recall_micro,hit_rate,catalog_coverage
3,fusion_rrf,100,0.131375,0.218485,0.374782
4,fusion_rrf,300,0.191353,0.296965,0.483562
5,fusion_rrf,500,0.221286,0.327591,0.490012
12,fusion_no_category,100,0.096674,0.161061,0.319551
13,fusion_no_category,300,0.138248,0.211375,0.335517
14,fusion_no_category,500,0.162195,0.236259,0.335526


## 11. Export reports and metadata

Primary output: `{split}_fusion_rrf_top500.parquet`, including source flags for later LightGBM ranking. Additional output: raw source union Parquet, category candidates Parquet, CSV metrics, and one JSON report.

In [12]:
for s in splits:
    path=OUT/f'{s}_fusion_category_top200.parquet'
    con.execute(f'COPY {s}_fusion_source_category TO {q(path)} (FORMAT PARQUET)')
    assert path.is_file() and path.stat().st_size>0

metrics_df.to_csv(REPORT/'notebook08_fusion_metrics.csv',index=False)
source_df.to_csv(REPORT/'notebook08_source_metrics.csv',index=False)
union_df.to_csv(REPORT/'notebook08_union_metrics.csv',index=False)
ov_df.to_csv(REPORT/'notebook08_source_overlap.csv',index=False)
unique_df.to_csv(REPORT/'notebook08_unique_positive_hits.csv',index=False)
abl_df.to_csv(REPORT/'notebook08_category_ablation.csv',index=False)

def safe(x):
    if isinstance(x,dict):return {str(k):safe(v) for k,v in x.items()}
    if isinstance(x,(list,tuple)):return [safe(v) for v in x]
    if isinstance(x,(np.integer,)):return int(x)
    if isinstance(x,(np.floating,)):return None if not np.isfinite(x) else float(x)
    if isinstance(x,float) and not math.isfinite(x):return None
    return x
report={'notebook':'08_multi_source_retrieval_fusion',
        'sources':['time_decayed_popularity_top100','item_cf_cooccurrence_top100',
                   'faiss_ivf_np16_top100','category_aware_top200'],
        'category_data_lineage':['item_properties_part1.csv','item_properties_part2.csv','category_tree.csv','events.csv'],
        'category_time_rule':'Use Notebook 02 split_item_features from latest property values with property_ts <= split cutoff; user-category counts from 30-day pre-cutoff history',
        'rrf_k':RRF_K,'weights':WEIGHTS,'max_k':TOP_K,'category_per_user':CATEGORY_PER_USER,
        'fallback':'Append historical time-decayed popularity after RRF primary candidates, excluding seen and duplicates; not an RRF vote',
        'runs':RUNS,'metrics':METRICS,'source_metrics':SOURCE_METRICS,'unranked_union':UNION_METRICS,
        'source_overlap':OVERLAPS,'unique_positive_hits':UNIQUE_HITS,'category_ablation':ABLATION,
        'pit_checks':PIT_CHECKS,'sanity':SANITY,
        'limitations':['Four primary sources have unequal candidate depths; union is unranked and is not Recall@500',
                       'RRF weights are fixed heuristics, not optimized; no causal improvement claims',
                       'Fallback is historical popularity, not personalized; users with no source still get recommendations',
                       'Category hierarchy is static metadata; item category assignment uses split point-in-time snapshot',
                       'Some historical interactions may precede their category assignment, but category snapshot never exceeds cutoff',
                       'All positive denominators are capped eligible future positives from Notebook 02, not full-future ground truth',
                       'No impressions or real CTR/CVR; no real online p95 latency',
                       'Only validation should guide future tuning; test is confirmatory']}
report_path=REPORT/'notebook08_fusion_report.json'
report_path.write_text(json.dumps(safe(report),indent=2,allow_nan=False),encoding='utf-8')
print('Report:',report_path)
print('Top-500 files:',[f'{s}_fusion_rrf_top500.parquet' for s in splits])


Report: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\reports\notebook08_fusion_report.json
Top-500 files: ['train_fusion_rrf_top500.parquet', 'validation_fusion_rrf_top500.parquet', 'test_fusion_rrf_top500.parquet']


## 12. Handoff to Notebook 09 — Feature Engineering

Upload `data/reports/notebook08_fusion_report.json` for interpretation. Notebook 09 will assemble **point-in-time** user, item, category, interaction, and retrieval-source features for the fused candidate pairs, with explicit leakage tests and missing-value handling. Do not train the ranker using `test` labels.
